# Agente Orientador de Discentes do BTI

## IMPLEMENTAÇÃO

### CONFIGURAÇÃO E IMPORTAÇÃO DE MÓDULOS

In [1]:
!pip install -q "agentkit @ git+https://github.com/silvaan/agentic-ai"

ERROR: Ignored the following versions that require a different python version: 0.2.0 Requires-Python >=3.9,<3.13; 0.2.1 Requires-Python >=3.9,<3.13; 0.2.3 Requires-Python >=3.9,<3.13; 1.0.0 Requires-Python >=3.9,<3.13; 1.0.1 Requires-Python >=3.9,<3.13; 1.0.2 Requires-Python >=3.9,<3.13; 1.0.3 Requires-Python >=3.9,<3.13; 1.0.4 Requires-Python >=3.9,<3.13; 1.1.0 Requires-Python >=3.9,<3.13; 1.1.1 Requires-Python >=3.9,<3.13; 1.2.0 Requires-Python >=3.9,<3.13; 1.2.1 Requires-Python >=3.9,<3.13; 1.2.10 Requires-Python >=3.10,<3.14; 1.2.11 Requires-Python >=3.10,<3.14; 1.2.12 Requires-Python >=3.10,<3.14; 1.2.13 Requires-Python >=3.10,<3.14; 1.2.2 Requires-Python >=3.9,<3.13; 1.2.3 Requires-Python >=3.9,<3.13; 1.2.4 Requires-Python >=3.9,<3.13; 1.2.5 Requires-Python >=3.9,<3.13; 1.2.6 Requires-Python >=3.9,<3.13; 1.2.7 Requires-Python >=3.9,<3.13; 1.2.8 Requires-Python >=3.9,<3.13; 1.2.9 Requires-Python >=3.9,<3.13; 1.3.0 Requires-Python >=3.10,<3.14; 1.3.1 Requires-Python >=3.10,<3.14; 1

In [2]:
import os
import json
import urllib.request
import shutil
import subprocess
from pathlib import Path

from pydantic import BaseModel, Field

from agentkit.model import LLMAPI
from agentkit.agent import Agent
from agentkit.tools import tool, run_python

ModuleNotFoundError: No module named 'agentkit'

In [ ]:
# import os
# from google.colab import userdata

# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
# os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
# os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
# os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [ ]:
llm = LLMAPI("gemini-3.5-flash-lite", api_key=userdata.get("GOOGLE_API_KEY"), base_url="https://generativelanguage.googleapis.com/v1beta/openai/", temperature=0.0, max_tokens=100)

### DEFINIÇÃO DE TOOLS

In [ ]:
@tool
def carrega_dados_disciplinas(): # -> list:
  pass

@tool
def classifica_intencao_usuario():
  pass

@tool
def recomenda_grade_disciplinas():
  pass

@tool
def coleta_feedback():
  pass

@tool
def calcula_carga_hora_disciplinas():
  pass

### CLASSES E FUNÇÕES DE APOIO

In [ ]:
class Planejamento(BaseModel):
  passos: list[str] = Field(min_length=2)

PLANNER_SYSTEM = """Você escreve planos para um agente que: 
a) recebe solicitações e dúvidas de alunos de uma universidade relacionadas a como montar a grade curricular do semestre baseado em critérios de interesse por disciplinas, pré-requisitos atendidos, disponibilidade e preferência do aluno em relação ao horário; 
b) consulta bases de dados; 
c) recebe feedback sobre o planejamento personalizado de grade curricular criado para o aluno;
d) cria novas sugestões de planejamento até o aluno concordar ou desistir das propostas.

O agente comprova o que escreve com testes automatizados, então o plano prevê escrever os testes e rodá-los.
Ainda não escreva código
"""

def cria_planejamento(tarefa: str) -> Planejamento:
  """Pede o plano ao modelo, com o papel no sistema e a tarefa no usuário."""
  return llm.generate_structured(
        [{"role": "system", "content": PLANNER_SYSTEM}, {"role": "user", "content": tarefa}],
        Planejamento,
        max_tokens=200,
    )

In [ ]:
TAREFA = """Planeje uma carga horária de {quantidade_horas} horas (ex: 240h) considerando que o aluno possui interesse por estas disciplinas: 
{lista de disciplinas} (ex: "Cálculo I, Introdução às Técnicas de Programação, Estruturas de Dados Básica I etc") e preferência por grades com
disciplinas sequenciais (ex: 24T34, 24T56) em vez de 24M12 e 24T12 (sugira apenas como última opção)
"""

planejamento = cria_planejamento(TAREFA)

for number, step in enumerate(planejamento.steps, start=1):
    print(f"{number}. {step}")